# Customer Segmentation Analysis using K-Means Clustering

## Objective
Segment customers based on purchasing behaviour using RFM (Recency, Frequency, Monetary) analysis and K-Means clustering to support targeted marketing strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

In [ ]:
df = pd.read_csv('OnlineRetail.csv', encoding='latin1')
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())

## Observation
The dataset contains a large number of retail transactions and includes customer IDs, invoice dates, quantities, prices and product information. Missing customer IDs and duplicate records must be handled before RFM analysis.

In [ ]:
df = df.dropna(subset=['CustomerID'])
df = df.drop_duplicates()
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
print('Cleaned shape:', df.shape)

## Observation
Missing customer IDs were removed because customer-level segmentation requires a customer identifier. Duplicate, cancelled and invalid transactions were excluded so that RFM metrics represent completed purchases.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

## RFM Feature Engineering
- **Recency:** days since the customer's most recent purchase.
- **Frequency:** number of unique invoices/purchases.
- **Monetary:** total customer spending.

In [ ]:
display(rfm.describe())

## Observation
Low Recency indicates recent activity, while higher Frequency and Monetary values indicate stronger purchasing engagement and customer value.

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

## Observation
StandardScaler was applied so that Recency, Frequency and Monetary are placed on comparable scales before K-Means clustering.

In [ ]:
wcss = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(rfm_scaled)
    wcss.append(model.inertia_)
plt.figure(figsize=(8,5))
plt.plot(range(1,11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS')
plt.show()

## Observation
The elbow is around K = 3, where the reduction in WCSS begins to flatten. Therefore, three clusters are used for the final segmentation.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
rfm.head()

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=rfm, x='Cluster')
plt.title('Number of Customers in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.show()

## Observation
The customer-count chart shows how the customer base is distributed across the three behavioural segments.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster', s=70)
plt.title('Customer Segments: Frequency vs Monetary')
plt.xlabel('Frequency')
plt.ylabel('Monetary Value')
plt.show()

## Observation
Customers with high purchase frequency and monetary value represent the strongest-value portion of the customer base.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', s=70)
plt.title('Customer Segments: Recency vs Monetary')
plt.xlabel('Recency')
plt.ylabel('Monetary Value')
plt.show()

## Observation
Low Recency combined with high Monetary value indicates highly active, valuable customers, while high Recency and low spending suggests customers who may need re-engagement.

In [ ]:
cluster_profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()
display(cluster_profile)

# Cluster Profiles and Marketing Recommendations

### Cluster 0 – Inactive / At-Risk Customers
Use win-back emails, targeted coupons and product recommendations to encourage another purchase.

### Cluster 1 – Regular / Loyal Customers
Use loyalty rewards, bundle offers and personalized recommendations to increase repeat purchases.

### Cluster 2 – VIP / High-Value Customers
Provide exclusive offers, early access, premium support and VIP rewards to retain these customers.

# Conclusion

RFM analysis and K-Means clustering divided the customer base into three behaviour-based segments. The segmentation enables differentiated marketing: re-engage inactive customers, strengthen loyalty among regular customers, and protect the highest-value customer relationships with VIP treatment.